# COGS 188 - Project Proposal

# Project Description

You have the choice of doing either (1) an AI solve a problem style project or (2) run a Special Topics class on a topic of your choice.  If you want to do (2) you should fill out the _other_ proposal for that. This is the proposal description for (1).

You will design and execute a machine learning project. There are a few constraints on the nature of the allowed project. 
- The problem addressed will not be a "toy problem" or "common training students problem" like 8-Queens or a small Traveling Salesman Problem or similar
- If its the kind of problem (e.g., RL) that interacts with a simulator or live task, then the problem will have a reasonably complex action space. For instance, a wupus world kind of thing with a 9x9 grid is definitely too small.  A simulated mountain car with a less complex 2-d road and simplified dynamics seems like a fairly low achievement level.  A more complex 3-d mountain car simulation with large extent and realistic dynamics, sure sounds great!
- If its the kind of problem that uses a dataset, then the dataset will have >1k observations and >5 variables. I'd prefer more like >10k observations and >10 variables. A general rule is that if you have >100x more observations than variables, your solution will likely generalize a lot better. The goal of training an unsupervised machine learning model is to learn the underlying pattern in a dataset in order to generalize well to unseen data, so choosing a large dataset is very important.
- The project must include some elements we talked about in the course
- The project will include a model selection and/or feature selection component where you will be looking for the best setup to maximize the performance of your AI system. Generally RL tasks may require a huge amount of training, so extensive grid search is unlikely to be possible. However expoloring a few reasonable hyper-parameters may still be possible. 
- You will evaluate the performance of your AI system using more than one appropriate metric
- You will be writing a report describing and discussing these accomplishments


Feel free to delete this description section when you hand in your proposal.

# Names
- Eric Nguyen
- Miles Davis
- Barry Tunstall

# Abstract 
The goal of this project is to build and compare AI agents on how they would play the board game 7 Wonders, which is a 3-7 player game. There are multiple factors that go into consideration when playing the game such as hidden hands from opponents, simulatenous selection from players, and delayed scoring which in all makes it so that solving the game through brute force would be impractical. Since there is no fixed dataset that exists for the task, we are going to be implementing the base game through code with a multiple agent environment that will generate us all the data needed through simulated play. Each observation would be a game state (the agent's hand, resources, coins, and visible board from opponents) which would also include the action taken by the agent along with the  final score. Using this environment that we created, we will compare four agents: a random baseline, a greedy heuristic benchmark, a heuristic agent who evaluation weights are tuned through simulated annealing, and a Monte Carlo search agent. Performance will be measured over a tournament in the style of round robin with thousands of simulated games using key metrics such as win rate, mean final score, and score margin.

# Background

Fill in the background and discuss the kind of prior work that has gone on in this research area here. **Use inline citation** to specify which references support which statements.  You can do that through HTML footnotes (demonstrated here). I used to reccommend Markdown footnotes (google is your friend) because they are simpler but recently I have had some problems with them working for me whereas HTML ones always work so far. So use the method that works for you, but do use inline citations.

Here is an example of inline citation. After government genocide in the 20th century, real birds were replaced with surveillance drones designed to look just like birds<a name="lorenz"></a>[<sup>[1]</sup>](#lorenznote). Use a minimum of 3 to 5 citations, but we prefer more <a name="admonish"></a>[<sup>[2]</sup>](#admonishnote). You need enough citations to fully explain and back up important facts. 

Remeber you are trying to explain why someone would want to answer your question or why your hypothesis is in the form that you've stated. 

# Possible Literatures

- Robilliard et. al: "Monte-Carlo Tree Search for the Game of '7 Wonders'." (https://doi.org/10.1007/978-3-319-14923-3_5)
- Paolini et. al: "Learning to Play 7 Wonders Duel Without Human Supervision." (https://arxiv.org/html/2406.00741v1)
- Jardim et. al: "An implementation of the 7 Wonders board game
for AI-based players." (https://www.sbgames.org/proceedings2020/ComputacaoShort/208553.pdf)
- Minenko, Igor. "Analysis and Development of Artificial Intelligence Based on Monte Carlo Tree Search for a Card Tabletop Game '7 Wonders'." (https://dspace.cuni.cz/handle/20.500.11956/200807)


# Problem Statement

Clearly describe the problem that you are solving. Avoid ambiguous words. The problem described should be well defined and should have at least one ML-relevant potential solution. Additionally, describe the problem thoroughly such that it is clear that the problem is quantifiable (the problem can be expressed in mathematical or logical terms), measurable (the problem can be measured by some metric and clearly observed), and replicable (the problem can be reproduced and occurs more than once).

# Proposed Solution

Since no dataset exists for this task, we'll build a multi-agent environment that implements the base game in Python and generates all of our data through simulated play. Each turn, every agent gets an observation (its hand, resources, and coins plus its opponents' visible boards) and a list of legal actions, everyone chooses at once, and the engine resolves the turn and passes the hands. As mentioned in our Ethics section, the engine automatically picks the cheapest way to pay for a card, so an action is just a card plus whether to build it, use it for a wonder stage, or sell it. We're writing our own engine instead of reusing an existing open-source one like Jardim et al.'s<a name="jardim"></a>[<sup>[1]</sup>](#jardimnote) so the Monte Carlo agent can copy and simulate game states directly. Every game is seeded so it can be replayed, and each turn's observation and action is logged along with that player's final score.

**Random baseline and greedy benchmark.** The random baseline picks uniformly from its legal actions. The greedy heuristic benchmark picks the action that maximizes a hand-weighted evaluation $V_w(s) = w^\top \phi(s)$, where $\phi(s)$ holds features like current points, resources produced, coins, military strength versus each neighbor, and wonder stages built.

**Tuned heuristic agent.** This agent uses the same features and decision rule, but its weights are tuned with simulated annealing<a name="kirkpatrick"></a>[<sup>[2]</sup>](#kirkpatricknote), so the tuning is the only difference from the benchmark. Each step adds Gaussian noise to the weights and scores the candidate $w'$ by its mean score margin $f(w')$ over $K$ games against random and greedy opponents, always keeping better candidates and keeping worse ones with probability $e^{(f(w') - f(w))/T}$ as the temperature $T$ cools. We'll experiment with the starting temperature, cooling rate, step size, and $K$.

**Monte Carlo search agent.** For each legal action, this agent runs simulations that fill its opponents' hands with random unseen cards (determinization<a name="cowling"></a>[<sup>[3]</sup>](#cowlingnote)) and then play the game to the end with a fast rollout policy for every player, picking the action with the best average result. This deals with hidden hands, simultaneous selection, and delayed scoring directly. We'll start with flat Monte Carlo and random rollouts, then try greedy rollouts or a full MCTS tree<a name="browne"></a>[<sup>[4]</sup>](#brownenote) if time allows, tuning the number of simulations per move along the way.

**Why it should work and how we'll test it.** Monte Carlo search doesn't need a hand-written evaluation function and has already beaten rule-based players in this game<a name="robilliard"></a>[<sup>[5]</sup>](#robilliardnote). Robilliard and Fonlupt also found that players evolved with genetic programming could compete with MCTS in 7 Wonders<a name="fonlupt"></a>[<sup>[6]</sup>](#fonluptnote), which suggests a well-tuned heuristic could come close at a much lower cost. Before the tournament we'll unit test the scoring and payment rules with hand-worked examples and check that random agents in every seat each win about $1/N$ of games. Then the agents face off in the round-robin tournament from the Evaluation Metrics section, with all code and data in our GitHub repo.

# Evaluation Metrics

We'll compare the four agents in a round-robin tournament of thousands of simulated games, with the greedy heuristic as our benchmark and results reported separately for each player count from 3 to 7. In head-to-head games one copy of an agent plays against $N-1$ copies of another, and in mixed games all four agents share a 4-player table. Seats will be rotated between agents with random wonder boards and matchups will use the same seeds.

**Win rate** (main metric): the fraction of games agent $a$ finishes first, $\text{WR}_a = \frac{1}{G}\sum_{g=1}^{G} w_{a,g}$, where $w_{a,g} = 1$ for a win and ties are broken by coins like in the real rules. Chance level is $1/N$, so anything clearly above that means an agent is stronger than its opponents.

**Mean final score**: average victory points per game, $\bar{s}_a = \frac{1}{G}\sum_{g} s_{a,g}$, also broken down by category (science, military, etc.) to see how each agent wins.

**Score margin**: points ahead of or behind the best opponent, $m_{a,g} = s_{a,g} - \max_{j \neq a} s_{j,g}$, averaged over games. This should tell whether a game is a blowout or a close vicotry/defeat, and because it's less noisy we'll also use it as the objective for simulated annealing.

**Time per move**: average seconds per decision. This shows what the Monte Carlo agent's extra strength costs, so we'll run it at a few simulation budgets and compare it to the tuned heuristic agent, which should be close to instant during games.

The heuristic's weights and the Monte Carlo settings will be tuned on one set of seeds and evaluated on new ones, and we'll report 95% confidence intervals.

# Ethics & Privacy

Our project has few direct privacy concerns. Since all of our data will be coming from simulated games from our own AI agents we will collect no information about real people. We will be not be using game logs from human players or from online platforms. If we do decide to add human gameplay data we would then only use publicly available and released datasets with the terms that this would be for research use and player identifiers being removed.

The ethical risk we might face is misuse of the AI agent. With a strong 7 Wonders agent, the agent could be used to cheat in online games against human players which is unfair and would violate the terms of service of most online game platforms. To mitigate this risk on our end we will make it so that our code will only run inside our own simulated environment and that there is no interface to any online platform. Our repository and README will state that the project is intended for academic research purposes only.

We also have the responsibility to report our results that we get from our agents honestly. Our agents will only be tested and compared against each other and not against human players and our engine will simplify parts of the game such as automatically choosing the cheapest way to pay for a card. A high win rate or score in our simulated tournaments therefore does not mean an agent will play well against real people. We will state these limitations and test our engine's rules carefully.

# Team Expectations 

* Communication: For scheduling and messaging we will being using Discord where we post updates and questions for each other. This is helpful as if anyone hits a blocker or a bug/agent that isn't working we could update each other promptly instead of having to wait for the next meeting to discuss
* Meetings: We will meet atleast twice a week over Discord starting in the early stages of our project so that we could ramp up quickly up to pace but then transistion to a weekly/as needed basis. In our meetings, each member will present their assigned work/parts and clearly update the team on any blockers, progress, or future state 
* Code/One Source of Truth: It is expected that all code lives on our repository on Github and that each team member will work on their own branch and merge through pull requests that the other team members review 
* Workload and Ownership: Work is going to be split evenly and each member will own and claim responsibility for their own part/section
* Decisions: Major decisions that invovle pivoting or changing our scope could be brought up in meetings or a as needed meeting in case of urgency
* Deadlines & falling behind: Anyone who expects to miss a deadline or checkpoint should notify the group ahead of time so that we could plan out how to tackle the problem and if there is someone falling behind repeatedly we will address that in a meeting

# Project Timeline Proposal

| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 11/01  |  5 PM |  Brainstorm some ideas/decisions for the project  | Introduced some initial ideas and then talked about the best times to meet/communicate | 
| 11/04  |  5 PM | Write designated proposal sections | Edited and Finalized proposal decisions | 
| 11/08  |  5 PM  | Encode game cards and wonders as a data file. Sketch game state, turn loop  | Agreement of engine interface that the agents will use   |
| 11/14  | 5 PM  | Agents could play full games with correct scoring and random/greedy agents implemented with some tests | Validate rules set up/fix bugs, discuss what to do for the next implementation of agents  |
| 11/18  | 5 PM  | Have the first working versions of the simulated annealing and monte carlo agent | Review the early results and then choose what parameters to set |
| 11/25  | 5 PM  | Have the parameters set and running; outline final report | Plan the final round robin tournaments and assign each person report sections |
| 12/05  | 5 PM  | Draft up the Background, Methods, and Results section of the project | Discuss & write the Discussion and Conclusion  |
| 12/11  | 5 PM  | Full report draft complete & clean up the repository | Conduct final review and finalize edits  |
| 12/14  | Before 11:59 PM  | Complete the final project & wrap up any thing in progress & finalize reports | Turn in Final Project  |

# Footnotes
<a name="jardimnote"></a>1.[^](#jardim): Jardim, J., Bettker, R., Minini, P., Pereira, G., Acosta, J., & Assunção, J. (2020) An implementation of the 7 Wonders board game for AI-based players. *Proceedings of SBGames 2020*, pp. 387–390. https://www.sbgames.org/proceedings2020/ComputacaoShort/208553.pdf<br>
<a name="kirkpatricknote"></a>2.[^](#kirkpatrick): Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983) Optimization by Simulated Annealing. *Science*, 220(4598), 671–680. https://doi.org/10.1126/science.220.4598.671<br>
<a name="cowlingnote"></a>3.[^](#cowling): Cowling, P. I., Powley, E. J., & Whitehouse, D. (2012) Information Set Monte Carlo Tree Search. *IEEE Transactions on Computational Intelligence and AI in Games*, 4(2), 120–143. https://doi.org/10.1109/TCIAIG.2012.2200894<br>
<a name="brownenote"></a>4.[^](#browne): Browne, C. B., et al. (2012) A Survey of Monte Carlo Tree Search Methods. *IEEE Transactions on Computational Intelligence and AI in Games*, 4(1), 1–43. https://doi.org/10.1109/TCIAIG.2012.2186810<br>
<a name="robilliardnote"></a>5.[^](#robilliard): Robilliard, D., Fonlupt, C., & Teytaud, F. (2014) Monte-Carlo Tree Search for the Game of "7 Wonders". *Computer Games Workshop (CGW 2014)*, CCIS vol. 504, pp. 64–77. https://doi.org/10.1007/978-3-319-14923-3_5<br>
<a name="fonluptnote"></a>6.[^](#fonlupt): Robilliard, D., & Fonlupt, C. (2015) Towards Human-Competitive Game Playing for Complex Board Games with Genetic Programming. *Artificial Evolution (EA 2015)*, LNCS vol. 9554, pp. 123–135. https://doi.org/10.1007/978-3-319-31471-6_10<br>